# Week 15: the robustness sweep (A4)

What happens to the detectors when a call is processed: a clone re-encoded as MP3 or Opus,
played in a noisy or echoing room, sped up, slowed down or shifted in pitch, or a call
recorded by the receiving handset or passed through a second G.711 leg. The transforms, the
clips, the measures and the reading rules were written down on 1 November 2026, before
anything was transformed or scored (`docs/results/robustness_v1.md`); the report that
applies them is `src/reporting/robustness.py`, written before the scores.

| Step | What | Where it runs | Time |
|---|---|---|---|
| render | 1,344 clean clips under `none` and 14 transforms, each then through the bundle's G.711 line | CPU | ~40 min |
| score | the final recipe's adapter (seed 1234) and SSL-AASIST (seed 1234) on all 20,160 clips | one T4 | ~1 h |
| report | each transform against `none` on the same clips, paired over speakers | CPU | ~2 min |

**How to run.** Add Input: the datasets that hold the **clean** clips of the new speakers
(`score_ext.csv`), of IndicF5 (`score_cm07.csv`) and of the training pool
(`codemix_adapt_train.csv`, for the babble), the final recipe's weights
(`final_s1234_best.pt`) and the larger encoder's (`ssl_aasist_final_s1234_fp16.pt`).
Settings: **GPU T4 x2** (one is used), Internet **On**. Then **Save Version -> Save & Run All
(Commit)**. Section 2 stops before any GPU work if a clip or a weights file is missing, and
names it.

**Getting the results.** The notebook's **Output** -> `robustness_results.zip` -> Download:
the per-clip scores, `summary.json` and `summary.md` (the table for `robustness_v1.md`). The
transformed audio is not kept.

## 1. Settings

In [ ]:
BRANCH = "week15-mounika-voice-check"
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
PLAN = "data/manifests/robustness_plan.csv"
BABBLE = "data/manifests/codemix_adapt_train.csv"
WEIGHTS = {"adapter": "final_s1234_best.pt", "ssl": "ssl_aasist_final_s1234_fp16.pt"}
ZIP_NAME = "robustness_results.zip"
SCORE_ARGS = ["--device", "cuda", "--batch-size", "32", "--num-workers", "4"]

## 2. Setup: clone, the clean clips, the weights, checks

In [ ]:
import glob, os, shutil, subprocess, sys, time
from pathlib import Path

WORK = Path("/kaggle/working")
TEMP = Path("/kaggle/temp") if os.access("/kaggle", os.W_OK) else Path("/tmp/w15")
TEMP.mkdir(parents=True, exist_ok=True)
REPO = TEMP / "codemix-deepfake-detection"
ROOT = TEMP / "root_clean"      # the clean clips, by name
RENDER = TEMP / "robust"        # every transformed clip, one folder per transform
OUT = TEMP / "robustness_results"


def run(args):
    """Run a command in the repository, streaming its output to the log, and stop on failure."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    with open(OUT / "run_log.txt", "a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(args) + "\n")
        proc = subprocess.Popen(args, cwd=REPO, text=True, errors="replace",
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if "arn" not in line:
                print(line, end="", flush=True)
    if proc.wait():
        save()
        raise SystemExit(f"failed ({proc.returncode}): {' '.join(args)} -- see run_log.txt")


def save():
    """Zip the results so far (never audio, never weights) into /kaggle/working."""
    import zipfile
    tmp = WORK / (ZIP_NAME + ".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT.rglob("*")):
            if f.is_file() and f.suffix not in (".wav", ".pt"):
                zf.write(f, f.relative_to(OUT.parent))
    tmp.replace(WORK / ZIP_NAME)


os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
for attempt in range(3):
    shutil.rmtree(REPO, ignore_errors=True)
    if subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                       f"https://{REPO_HOST}", str(REPO)]).returncode == 0:
        break
    time.sleep(20)
else:
    raise SystemExit("Could not clone the repository. Is Internet switched On?")
sys.path.insert(0, str(REPO))
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True, capture_output=True).stdout.strip()
(OUT / "commit.txt").write_text(commit + "\n")
print("commit", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile==0.12.1", "librosa==0.11.0"])
if shutil.which("ffmpeg") is None:
    raise SystemExit("ffmpeg is missing: the codec, tempo and pitch transforms need it")

import pandas as pd
from src.data.scoring_manifests import missing_files

# the clean clips: every clips/ folder under a clean/ folder of an attached dataset
folders = [p for p in glob.glob("/kaggle/input/**/clean/**/clips", recursive=True) if os.listdir(p)]
folders += [p for p in glob.glob("/kaggle/input/**/clean/clips", recursive=True)
            if p not in folders and os.listdir(p)]
print("clean clip folders:", folders)
shutil.rmtree(ROOT, ignore_errors=True)
(ROOT / "clips").mkdir(parents=True)
for folder in folders:
    for name in os.listdir(folder):
        target = ROOT / "clips" / name
        if not target.exists():
            os.symlink(os.path.join(folder, name), target)
for manifest in (PLAN, BABBLE):
    frame = pd.read_csv(REPO / manifest)
    if manifest == BABBLE:
        frame = frame[frame["label"] == "bonafide"]
    gone = missing_files(frame, str(ROOT))
    print(f"{manifest}: {len(frame)} clips, {len(gone)} missing")
    if gone:
        raise SystemExit(f"{manifest} needs clean clips no attached dataset holds, e.g. {gone[:3]}")

weights = {}
for model, name in WEIGHTS.items():
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not found:
        raise SystemExit(f"Attach the dataset that holds {name} (Add Input).")
    weights[model] = found[0]
    print(model, "weights:", found[0])
run([sys.executable, "-m", "pytest", "-q", "tests/test_robustness.py", "tests/test_robustness_report.py"])
print("every clip and both weights are present; the tests pass")

## 3. Render, score, report, and the download link

In [ ]:
# 1. render: every plan clip under `none` and the 14 transforms (resumable)
run([sys.executable, "-m", "src.data.robustness", "render", "--plan", PLAN, "--babble", BABBLE,
     "--data-root", ROOT, "--out", RENDER])
shutil.copy(RENDER / "robustness_manifest.csv", OUT / "robustness_manifest.csv")
save()

# 2. score both models on every rendered clip
manifest = RENDER / "robustness_manifest.csv"
run([sys.executable, "-m", "src.training.evaluate", "--checkpoint", weights["adapter"],
     "--manifest", manifest, "--encoder", "wav2vec2-base", "--data-root", RENDER, *SCORE_ARGS,
     "--max-seconds", "4.0", "--scores-out", OUT / "adapter_scores.csv", "--out", OUT / "adapter.json"])
save()
run([sys.executable, "-m", "src.baselines.aasist", "score", "--weights", weights["ssl"], "--model", "ssl",
     "--manifest", manifest, "--data-root", RENDER, *SCORE_ARGS,
     "--scores-out", OUT / "ssl_scores.csv", "--out", OUT / "ssl.json"])
save()

# 3. the report, by the rules written before scoring
run([sys.executable, "-m", "src.reporting.robustness", "--scores", f"adapter={OUT / 'adapter_scores.csv'}",
     "--scores", f"ssl={OUT / 'ssl_scores.csv'}", "--out", OUT / "summary.json"])
(OUT / "summary.txt").write_text("STATUS: FINISHED - every step ran.\ncommit: " + commit + "\n")
save()
print((OUT / "summary.md").read_text())

from IPython.display import FileLink, display
os.chdir(WORK)
display(FileLink(ZIP_NAME))